In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from roar import ALL_VEHICLES, MIC_CHANNELS, MIC_CHANNELS_CLEANED
from roar import DATA_DIR as DATA_ROOT
from roar.preprocessing import (
    extract_features_from_h5_file,
    fix_channel_names,
    get_channel_mapping,
    parse_filename,
)

device = (
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)

### Settings

In [ ]:
# LOGO_COLUMN = "vehicle"
LOGO_COLUMN = "measure"

### Mappings

In [3]:
TRACKS = ["track150", "track211"]
CARS = ALL_VEHICLES
TYRES = ["tyre1", "tyre3", "tyre6", "tyre10", "tyre12", "tyre13"]

In [4]:
# Set the data root
if not DATA_ROOT.exists():
    DATA_ROOT = Path(r"C:\Users\Lars\Büro\KIT\Master\WS_25_26\AIFB_Seminar\projects\workspace\data")

SYNONYM_TO_CHANNEL = get_channel_mapping()

h5_paths = list(DATA_ROOT.rglob("*.h5"))  # pyright: ignore[reportPossiblyUnboundVariable]
print(f"Found {len(h5_paths)} files")

for h5_path in h5_paths:
    fix_channel_names(file_path=h5_path, mapping=SYNONYM_TO_CHANNEL, verbose=False)

Found 204 files


### Build dataset

In [5]:
from tqdm import tqdm

# Get the h5 file paths
rows = []
for path in tqdm(h5_paths):
    feat_dict = extract_features_from_h5_file(
        path, channels=MIC_CHANNELS + MIC_CHANNELS_CLEANED + ["speed"]
    )
    meta = parse_filename(path)

    row = {
        **feat_dict,
        **meta,
    }
    rows.append(row)

df = pd.DataFrame(rows)

# df.info(verbose=True, show_counts=True)

  0%|          | 0/204 [00:00<?, ?it/s]

100%|██████████| 204/204 [02:08<00:00,  1.58it/s]


### Fix missing values

1. mic_iso channels 151/204 non-null

    track211 ID.4 tyre3 is missing mic iso, remove it completely for now, contains however "MikrofonOst" & "MikrofonWest" with similar sample rates (btw regular files also contain a "mic_2m" additionally to "mic_iso") which could be used.  
    These channels are actually what we need -> this problem is fixed, in fix_channel_names this is taken care of

2. NAWSSound channels 199/204 non-null

    5 track211 Q8 e-tron tyre12 files are missing the NAWS channel, fill it with means of the respective group

In [6]:
df_clean = df.copy()

# Fill missing NAWSSound features with group-wise mean imputation
missing_cols = [col for col in df_clean.columns if col.startswith("NAWSSound")]
group_cols = ["track_ID", "vehicle", "tyre_ID"]

for col in missing_cols:
    df_clean[col] = df_clean.groupby(group_cols)[col].transform(lambda g: g.fillna(g.mean()))

# If some groups have only NaN (rare), fill remaining with global mean
if df_clean[missing_cols].isna().any().any():
    print("Some NAWSSound groups have only NaN, filling with global mean.")
    df_clean[missing_cols] = df_clean[missing_cols].fillna(df_clean[missing_cols].mean())

# df_clean.info(verbose=True, show_counts=True)

### Transform to Categorical

In [7]:
from roar import MEASUREMENTS, MEASUREMENTS_CLEAN_NAMES

measurement_type = pd.CategoricalDtype(categories=MEASUREMENTS, ordered=True)
df_clean["measure"] = df_clean["measure"].replace(MEASUREMENTS_CLEAN_NAMES).astype(measurement_type)

df_clean["track_ID"] = df_clean["track_ID"].astype("category")

vehicle_type = pd.CategoricalDtype(categories=ALL_VEHICLES, ordered=True)
df_clean["vehicle"] = df_clean["vehicle"].astype(vehicle_type)

### Create LOGO Split

In [8]:
# Explicitly exclude non-feature columns (safer than dtype selection alone)
non_feature_cols = ["track_ID", "vehicle", "tyre_ID", "file_stem", "file_path", "measure", "date"]
feature_cols = [
    c for c in df_clean.columns if c not in non_feature_cols and df_clean[c].dtype == "float64"
]

print("Number of features:", len(feature_cols))

X = df_clean[feature_cols]
y = df_clean["track_ID"]
groups = df_clean[LOGO_COLUMN]

# Encode string labels to integers for XGBoost
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Classes:", le.classes_)

# LOGO splitter
logo = LeaveOneGroupOut()

split = logo.split(X, y_enc, groups)
print("LOGO splits and test groups:")
for i, (_, test_idx) in enumerate(list(split)):
    print(i, np.unique(groups[test_idx]))

Number of features: 1438
Classes: [150 211]
LOGO splits and test groups:
0 ['meas1']
1 ['meas2']
2 ['meas3']
3 ['meas4']
4 ['meas5']
5 ['meas6']


### Feature Selection Transformer
This lets us select different feature subsetis during the gridsearch

In [9]:
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureSubsetSelector(BaseEstimator, TransformerMixin):
    def __init__(self, feature_set=None, feature_sets=None):
        self.feature_set = feature_set
        self.feature_sets = feature_sets
        if self.feature_sets is None:
            raise ValueError("feature_sets must be provided")

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if self.feature_sets is None:
            raise ValueError("feature_sets must be provided")
        features = self.feature_sets[self.feature_set]
        return X[features]

In [10]:
# additional control features such as average speed, acceleration, ...
speed_accel_features = [
    feature for feature in feature_cols if "speed" in feature or "accel" in feature
]
# Define the Feature subsets we will use later
base_features = [
    feature
    for feature in feature_cols
    if "_cleaned" not in feature and feature not in speed_accel_features
]
clean_features = [
    feature
    for feature in feature_cols
    if "_cleaned" in feature and feature not in speed_accel_features
]
invariant_features = [
    feature for feature in feature_cols if "_invariant" in feature and "_cleaned" not in feature
]
clean_invariant_features = [
    feature for feature in feature_cols if "_cleaned" in feature and "_invariant" in feature
]

FEATURE_SETS = {
    "basic": base_features,
    "cleaned": clean_features,
    "cleaned+speed": clean_features + speed_accel_features,
    "invariant": invariant_features,
    "cleaned_invariant": clean_invariant_features,
    "cleaned_invariant+speed": clean_invariant_features + speed_accel_features,
}

### Wrapper to adapt the threshold
Works if model has predict_proba so we can tune that on the validation set

In [11]:
from sklearn.base import ClassifierMixin, clone


class ThresholdClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold
        self.response_method = "predict_proba"

    def fit(self, X, y) -> "ThresholdClassifier":
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y)
        return self

    def predict_proba(self, X) -> np.ndarray:
        return self.estimator_.predict_proba(X)

    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

### Some more setup to collect all results

In [12]:
# At the top of your file or module
RESULTS_PATTERN = r"^(param_feature_selector__feature_set|split\d+_test_(weighted_f1|acc)|(mean|std|rank)_test_(weighted_f1|acc))$"


def add_results_to_df(grid_search, res_df=None, model_name=None):
    cv_results = pd.DataFrame(grid_search.cv_results_)
    relevant_cols = cv_results.filter(regex=RESULTS_PATTERN).columns.tolist()

    filtered_results = cv_results[relevant_cols].copy()
    filtered_results["model"] = model_name

    if res_df is None:
        return filtered_results

    res_df = pd.concat([res_df, filtered_results], ignore_index=True)
    return res_df

In [13]:
# Reset results dataframe
res_df = None

### Dummy regressor

In [14]:
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [16]:
pipeline_dummy = Pipeline(
    [
        ("feature_selector", FeatureSubsetSelector(feature_sets=FEATURE_SETS)),
        (
            "clf",
            DummyClassifier(strategy="most_frequent"),
        ),
    ]
)

param_grid = {
    "feature_selector__feature_set": list(FEATURE_SETS.keys()),
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=pipeline_dummy,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    error_score="raise",
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)

res_df = add_results_to_df(grid, res_df=res_df, model_name="DummyClassifier")

Fitting 4 folds for each of 6 candidates, totalling 24 fits


### Logistic Regression with PCA

In [15]:
pipeline = Pipeline(
    [
        ("feature_selector", FeatureSubsetSelector(feature_sets=FEATURE_SETS)),
        ("scaler", StandardScaler()),  # standardize features
        ("pca", PCA(n_components=0.95)),  # reduce to top 5 PCs
        (
            "clf",
            ThresholdClassifier(
                estimator=LogisticRegression(random_state=42, max_iter=1000), threshold=0.5
            ),
        ),
    ]
)

param_grid = {
    "feature_selector__feature_set": list(FEATURE_SETS.keys()),
    "pca__n_components": [0.8, 0.9, 0.95, 0.99, 0.999],
    "clf__threshold": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "clf__estimator__C": [0.1, 1, 10],
    "clf__estimator__class_weight": [None, "balanced"],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    error_score="raise",
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)

pd.DataFrame(grid.cv_results_).to_csv(f"log_reg_logo_{LOGO_COLUMN}_results.csv", index=False)

# res_df = add_results_to_df(grid, res_df=res_df, model_name="LogisticRegression_PCA")

pd.DataFrame(grid.cv_results_)[
    [
        "param_feature_selector__feature_set",
        "param_pca__n_components",
        "param_clf__threshold",
        "param_clf__estimator__C",
        "param_clf__estimator__class_weight",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 6 folds for each of 1260 candidates, totalling 7560 fits


,param_feature_selector__feature_set,param_pca__n_components,param_clf__threshold,param_clf__estimator__C,param_clf__estimator__class_weight,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
943,cleaned+speed,0.990,0.5,10.0,None,0.965196,0.049221,0.965217,0.049254
554,cleaned+speed,0.999,0.6,1.0,None,0.964697,0.046724,0.965507,0.044698
1093,cleaned+speed,0.990,0.3,10.0,balanced,0.964634,0.046706,0.965507,0.044698
1038,invariant,0.990,0.8,10.0,None,0.964078,0.035294,0.960507,0.033967
1218,invariant,0.990,0.7,10.0,balanced,0.964078,0.035294,0.960507,0.033967


### Random Forest

In [18]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
)
threshold_rf = ThresholdClassifier(estimator=rf, threshold=0.5)

pipeline_rf = Pipeline(
    [
        ("feature_selector", FeatureSubsetSelector(feature_sets=FEATURE_SETS)),
        ("clf", threshold_rf),
    ]
)

param_grid = {
    "feature_selector__feature_set": list(FEATURE_SETS.keys()),
    "clf__threshold": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "clf__estimator__max_depth": [None, 5, 10],
    "clf__estimator__min_samples_leaf": [1, 5, 10],
    "clf__estimator__class_weight": [None, "balanced", "balanced_subsample"],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)

pd.DataFrame(grid.cv_results_).to_csv(f"rf_logo_{LOGO_COLUMN}_results.csv", index=False)

res_df = add_results_to_df(grid, res_df, model_name="RandomForest")

pd.DataFrame(grid.cv_results_)[
    [
        "param_feature_selector__feature_set",
        "param_clf__estimator__max_depth",
        "param_clf__estimator__min_samples_leaf",
        "param_clf__estimator__class_weight",
        "param_clf__threshold",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 1134 candidates, totalling 4536 fits


,param_feature_selector__feature_set,param_clf__estimator__max_depth,param_clf__estimator__min_samples_leaf,param_clf__estimator__class_weight,param_clf__threshold,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
26,cleaned+speed,None,1,None,0.6,0.728000,0.157944,0.772122,0.103332
278,cleaned+speed,10,1,None,0.6,0.728000,0.157944,0.772122,0.103332
155,cleaned_invariant+speed,5,1,None,0.6,0.726897,0.117828,0.760140,0.068834
29,cleaned_invariant+speed,None,1,None,0.6,0.726312,0.116769,0.758491,0.068126
281,cleaned_invariant+speed,10,1,None,0.6,0.726312,0.116769,0.758491,0.068126


### SVM

In [19]:
svc = SVC(random_state=42)

pipeline_svc = Pipeline(
    [
        ("feature_selector", FeatureSubsetSelector(feature_sets=FEATURE_SETS)),
        ("scaler", StandardScaler()),  # standardize features
        ("clf", svc),
    ]
)

param_grid = {
    "feature_selector__feature_set": list(FEATURE_SETS.keys()),
    "clf__kernel": ["rbf", "poly"],
    "clf__degree": [2, 3, 4],
    "clf__gamma": ["scale", "auto"],
    "clf__C": [0.1, 1, 10],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=pipeline_svc,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)

pd.DataFrame(grid.cv_results_).to_csv(f"svc_logo_{LOGO_COLUMN}_results.csv", index=False)

res_df = add_results_to_df(grid, res_df, model_name="SVC")

pd.DataFrame(grid.cv_results_)[
    [
        "param_feature_selector__feature_set",
        "param_clf__kernel",
        "param_clf__degree",
        "param_clf__gamma",
        "param_clf__C",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 216 candidates, totalling 864 fits


,param_feature_selector__feature_set,param_clf__kernel,param_clf__degree,param_clf__gamma,param_clf__C,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
175,cleaned,poly,3,scale,10.0,0.748982,0.119059,0.797894,0.095981
187,cleaned,poly,3,auto,10.0,0.748982,0.119059,0.797894,0.095981
176,cleaned+speed,poly,3,scale,10.0,0.730836,0.144965,0.788634,0.109085
188,cleaned+speed,poly,3,auto,10.0,0.730836,0.144965,0.788634,0.109085
178,cleaned_invariant,poly,3,scale,10.0,0.707769,0.103622,0.769216,0.077791


### XGBoost

In [20]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    n_jobs=-1,
    eval_metric="logloss",
    random_state=42,
)

threshold_xgb = ThresholdClassifier(estimator=xgb, threshold=0.5)

pipeline_xgb = Pipeline(
    [
        ("feature_selector", FeatureSubsetSelector(feature_sets=FEATURE_SETS)),
        ("clf", threshold_xgb),
    ]
)

param_grid = {
    "feature_selector__feature_set": list(FEATURE_SETS.keys()),
    "clf__threshold": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "clf__estimator__max_depth": [3, 5, 7],
    "clf__estimator__min_child_weight": [1, 5],
    "clf__estimator__learning_rate": [0.01, 0.05, 0.1],
    "clf__estimator__gamma": [0, 0.1],
    "clf__estimator__subsample": [0.8, 1.0],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)

pd.DataFrame(grid.cv_results_).to_csv(f"xgb_logo_{LOGO_COLUMN}_results.csv", index=False)

res_df = add_results_to_df(grid, res_df, model_name="XGBoost")

pd.DataFrame(grid.cv_results_)[
    [
        "param_feature_selector__feature_set",
        "param_clf__threshold",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 3024 candidates, totalling 12096 fits


,param_feature_selector__feature_set,param_clf__threshold,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
2523,invariant,0.2,0.727150,0.205816,0.786912,0.141890
2534,cleaned+speed,0.4,0.727119,0.206737,0.781473,0.140640
2197,cleaned,0.4,0.726240,0.203421,0.783613,0.136818
2365,cleaned,0.4,0.726240,0.203421,0.783613,0.136818
1226,cleaned+speed,0.3,0.725909,0.203105,0.783613,0.136818


### LightGBM

In [21]:
import warnings

warnings.filterwarnings("ignore", message="X does not have valid feature names*")

In [22]:
lgbm = LGBMClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

threshold_lgbm = ThresholdClassifier(estimator=lgbm, threshold=0.5)

pipeline_lgbm = Pipeline(
    [
        ("feature_selector", FeatureSubsetSelector(feature_sets=FEATURE_SETS)),
        ("clf", threshold_lgbm),
    ]
)

param_grid = {
    "feature_selector__feature_set": list(FEATURE_SETS.keys()),
    "clf__threshold": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "clf__estimator__num_leaves": [31, 50, 100],
    "clf__estimator__max_depth": [5, 10, 20],
    "clf__estimator__learning_rate": [0.01, 0.05, 0.1],
    "clf__estimator__class_weight": [None, "balanced"],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=pipeline_lgbm,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)

pd.DataFrame(grid.cv_results_).to_csv(f"lgbm_logo_{LOGO_COLUMN}_results.csv", index=False)

res_df = add_results_to_df(grid, res_df, model_name="LightGBM")

pd.DataFrame(grid.cv_results_)[
    [
        "param_feature_selector__feature_set",
        "param_clf__threshold",
        "param_clf__estimator__num_leaves",
        "param_clf__estimator__max_depth",
        "param_clf__estimator__learning_rate",
        "param_clf__estimator__learning_rate",
        "param_clf__estimator__class_weight",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 2268 candidates, totalling 9072 fits


,param_feature_selector__feature_set,param_clf__threshold,param_clf__estimator__num_leaves,param_clf__estimator__max_depth,param_clf__estimator__learning_rate,param_clf__estimator__learning_rate,param_clf__estimator__class_weight,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
1654,cleaned_invariant,0.4,31,10,0.05,0.05,balanced,0.75339,0.156047,0.790967,0.107519
1696,cleaned_invariant,0.4,50,10,0.05,0.05,balanced,0.75339,0.156047,0.790967,0.107519
1738,cleaned_invariant,0.4,100,10,0.05,0.05,balanced,0.75339,0.156047,0.790967,0.107519
1780,cleaned_invariant,0.4,31,20,0.05,0.05,balanced,0.75339,0.156047,0.790967,0.107519
1822,cleaned_invariant,0.4,50,20,0.05,0.05,balanced,0.75339,0.156047,0.790967,0.107519


### TabFPM

In [23]:
from tabpfn import TabPFNClassifier

# For this to work you must accept the license terms at https://huggingface.co/Prior-Labs/tabpfn_2_5
# and be logged in via the huggingface_hub CLI.

tabpfn = TabPFNClassifier()

pipeline_tabpfn = Pipeline(
    [
        ("feature_selector", FeatureSubsetSelector(feature_sets=FEATURE_SETS)),
        ("clf", tabpfn),
    ]
)

param_grid = {
    "feature_selector__feature_set": list(FEATURE_SETS.keys()),
    "clf__balance_probabilities": [True, False],
    "clf__n_estimators": [2, 4, 8, 16],
}

grid = GridSearchCV(
    estimator=pipeline_tabpfn,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=2,
)
grid.fit(X, y_enc)

pd.DataFrame(grid.cv_results_).to_csv(f"tabpfn_logo_{LOGO_COLUMN}_results.csv", index=False)

res_df = add_results_to_df(grid, res_df, model_name="TabPFN")

pd.DataFrame(grid.cv_results_)[
    [
        "param_feature_selector__feature_set",
        "param_clf__balance_probabilities",
        "param_clf__n_estimators",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 48 candidates, totalling 192 fits
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=basic; total time=  40.8s
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=cleaned; total time=  44.9s
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=cleaned; total time=  44.9s
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=basic; total time=  45.3s
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=cleaned; total time=  45.4s
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=basic; total time=  45.5s
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=cleaned; total time=  45.5s
[CV] END clf__balance_probabilities=True, clf__n_estimators=2, feature_selector__feature_set=basi

KeyboardInterrupt: 

## Visualize Results

In [ ]:
res_df.to_csv(f"{LOGO_COLUMN}_logo_results.csv", index=False)

### Balancing Helper

In [ ]:
def car_balanced_weights(vehicles):
    counts = Counter(vehicles)
    w = np.array([1.0 / counts[c] for c in vehicles], dtype=float)
    # normalize for nicer scales (optional)
    return w * (len(w) / w.sum())

### Training Loop

In [ ]:
# Config
CLASS_BALANCING = False
CAR_BALANCING = True
THRESHOLD_TUNING = True  # Only for Random Forest

In [ ]:
rf_accs, rf_f1s = [], []
xgb_accs, xgb_f1s = [], []
lgbm_accs, lgbm_f1s = [], []
tabpfn_accs, tabpfn_f1s = [], []
fold_results = []

# Optional: to accumulate global confusion matrices across folds
# (nice for seminar slides)
labels = le.classes_
rf_cm_total = np.zeros((len(labels), len(labels)), dtype=int)
xgb_cm_total = np.zeros_like(rf_cm_total)
lgbm_cm_total = np.zeros_like(rf_cm_total)
tabpfn_cm_total = np.zeros_like(rf_cm_total)

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups), start=1):
    held_out_car = groups[test_idx][0]
    print(f"\n===== LOGO Fold {fold} | Held-out car: {held_out_car} =====")
    print(f"Train n={len(train_idx)}, Test n={len(test_idx)}")

    # Split data (enc for XGBoost)
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    y_train_enc, y_test_enc = y_enc[train_idx], y_enc[test_idx]

    # Compute class weights
    class_weights = compute_sample_weight("balanced", y_train)

    # Compute car-based weights to balance training samples by car
    car_train = df_clean.iloc[train_idx]["vehicle"].to_numpy()
    car_weights = car_balanced_weights(car_train)

    # ------------------ Random Forest ------------------
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=42,
    )

    # Fit with appropriate sample weights
    if CLASS_BALANCING and CAR_BALANCING:
        combined_weights = class_weights * car_weights
        combined_weights = combined_weights * (len(combined_weights) / combined_weights.sum())
        rf.fit(X_train, y_train, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        rf.fit(X_train, y_train, sample_weight=class_weights)
    elif CAR_BALANCING:
        rf.fit(X_train, y_train, sample_weight=car_weights)
    else:
        rf.fit(X_train, y_train)

    # Threshold tuning
    if THRESHOLD_TUNING:
        proba0 = rf.predict_proba(X_test)[:, 0]
        t0 = 0.3  # threshold for class 0
        y_pred_rf_enc = (proba0 < t0).astype(int)
        y_pred_rf = le.inverse_transform(y_pred_rf_enc)
    else:
        y_pred_rf = rf.predict(X_test)

    rf_acc = accuracy_score(y_test, y_pred_rf)
    rf_f1 = f1_score(y_test, y_pred_rf, average="weighted")

    rf_accs.append(rf_acc)
    rf_f1s.append(rf_f1)

    cm_rf = confusion_matrix(y_test, y_pred_rf, labels=labels)
    rf_cm_total += cm_rf

    print(f"RF  - acc: {rf_acc:.3f}, f1: {rf_f1:.3f}")
    print(cm_rf)

    # ------------------ XGBoost ------------------
    # For 2-class problems we can use binary:logistic.
    # If we ever add more road types, switch to multi:softprob + num_class.
    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        n_jobs=-1,
        eval_metric="logloss",
        random_state=42,
    )

    if CLASS_BALANCING and CAR_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=class_weights)
    elif CAR_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=car_weights)
    else:
        xgb.fit(X_train, y_train_enc)

    y_pred_xgb_enc = xgb.predict(X_test)
    y_pred_xgb = le.inverse_transform(y_pred_xgb_enc.astype(int))

    xgb_acc = accuracy_score(y_test, y_pred_xgb)
    xgb_f1 = f1_score(y_test, y_pred_xgb, average="weighted")

    xgb_accs.append(xgb_acc)
    xgb_f1s.append(xgb_f1)

    cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=labels)
    xgb_cm_total += cm_xgb

    print(f"XGB - acc: {xgb_acc:.3f}, f1: {xgb_f1:.3f}")
    print(cm_xgb)

    # ------------------ LightGBM ------------------
    lgbm = LGBMClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )

    if CLASS_BALANCING and CAR_BALANCING:
        lgbm.fit(X_train, y_train_enc, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        lgbm.fit(X_train, y_train_enc, sample_weight=class_weights)
    elif CAR_BALANCING:
        lgbm.fit(X_train, y_train_enc, sample_weight=car_weights)
    else:
        lgbm.fit(X_train, y_train_enc)

    if THRESHOLD_TUNING:
        proba0 = lgbm.predict_proba(X_test)[:, 0]
        t0 = 0.3  # threshold for class 0
        y_pred_lgbm_enc = (proba0 < t0).astype(int)
        y_pred_lgbm = le.inverse_transform(y_pred_lgbm_enc)
    else:
        y_pred_lgbm_enc = lgbm.predict(X_test)
        y_pred_lgbm = le.inverse_transform(y_pred_lgbm_enc.astype(int))

    lgbm_acc = accuracy_score(y_test, y_pred_lgbm)
    lgbm_f1 = f1_score(y_test, y_pred_lgbm, average="weighted")

    lgbm_accs.append(lgbm_acc)
    lgbm_f1s.append(lgbm_f1)

    cm_lgbm = confusion_matrix(y_test, y_pred_lgbm, labels=labels)
    lgbm_cm_total += cm_lgbm

    print(f"LGBM - acc: {lgbm_acc:.3f}, f1: {lgbm_f1:.3f}")
    print(cm_lgbm)

    # ------------------ TabPFN ------------------
    tabpfn = TabPFNClassifier()
    tabpfn.fit(X_train, y_train_enc)

    if THRESHOLD_TUNING:
        proba0 = tabpfn.predict_proba(X_test)[:, 0]
        t0 = 0.3  # threshold for class 0
        y_pred_tabpfn_enc = (proba0 < t0).astype(int)
        y_pred_tabpfn = le.inverse_transform(y_pred_tabpfn_enc)
    else:
        y_pred_tabpfn_enc = tabpfn.predict(X_test)
    y_pred_tabpfn = le.inverse_transform(y_pred_tabpfn_enc.astype(int))

    tabpfn_acc = accuracy_score(y_test, y_pred_tabpfn)
    tabpfn_f1 = f1_score(y_test, y_pred_tabpfn, average="weighted")

    tabpfn_accs.append(tabpfn_acc)
    tabpfn_f1s.append(tabpfn_f1)

    cm_tabpfn = confusion_matrix(y_test, y_pred_tabpfn, labels=labels)
    tabpfn_cm_total += cm_tabpfn

    print(f"TabPFN - acc: {tabpfn_acc:.3f}, f1: {tabpfn_f1:.3f}")
    print(cm_tabpfn)

    fold_results.append(
        {
            "fold": fold,
            "held_out_car": held_out_car,
            "n_test": len(test_idx),
            "rf_acc": rf_acc,
            "rf_f1": rf_f1,
            "xgb_acc": xgb_acc,
            "xgb_f1": xgb_f1,
            "lgbm_acc": lgbm_acc,
            "lgbm_f1": lgbm_f1,
            "tabpfn_acc": tabpfn_acc,
            "tabpfn_f1": tabpfn_f1,
        }
    )


# --------------- Summary ---------------
print("\n===== LOGO Summary (per held-out car) =====")
for r in fold_results:
    print(
        f"{r['held_out_car']}: RF acc={r['rf_acc']:.3f}, XGB acc={r['xgb_acc']:.3f}, "
        f"LGBM acc={r['lgbm_acc']:.3f}, TabPFN acc={r['tabpfn_acc']:.3f}"
    )

print("\n===== Overall (mean ± std) =====")
print(
    f"RF  acc: {np.mean(rf_accs):.3f} ± {np.std(rf_accs):.3f} | "
    f"f1: {np.mean(rf_f1s):.3f} ± {np.std(rf_f1s):.3f}"
)
print(
    f"XGB acc: {np.mean(xgb_accs):.3f} ± {np.std(xgb_accs):.3f} | "
    f"f1: {np.mean(xgb_f1s):.3f} ± {np.std(xgb_f1s):.3f}"
)
print(
    f"LGBM acc: {np.mean(lgbm_accs):.3f} ± {np.std(lgbm_accs):.3f} | "
    f"f1: {np.mean(lgbm_f1s):.3f} ± {np.std(lgbm_f1s):.3f}"
)
print(
    f"TabPFN acc: {np.mean(tabpfn_accs):.3f} ± {np.std(tabpfn_accs):.3f} | "
    f"f1: {np.mean(tabpfn_f1s):.3f} ± {np.std(tabpfn_f1s):.3f}"
)

print("\nRF total confusion matrix (rows=true, cols=pred):")
print(rf_cm_total)
print("\nXGB total confusion matrix (rows=true, cols=pred):")
print(xgb_cm_total)
print("\nLGBM total confusion matrix (rows=true, cols=pred):")
print(lgbm_cm_total)
print("\nTabPFN total confusion matrix (rows=true, cols=pred):")
print(tabpfn_cm_total)
print("\nLabel order:", labels)

### Feature Importance

In [ ]:
rf_final = RandomForestClassifier(n_estimators=500, max_depth=None, n_jobs=-1, random_state=42)

# Compute class weights
class_weights_final = compute_sample_weight("balanced", y)

# Compute car-based weights to balance training samples by car
car_final = df_clean["vehicle"].to_numpy()
car_weights_final = car_balanced_weights(car_final)
if CLASS_BALANCING and CAR_BALANCING:
    combined_weights_final = class_weights_final * car_weights_final
    combined_weights_final = combined_weights_final * (
        len(combined_weights_final) / combined_weights_final.sum()
    )
    rf_final.fit(X, y, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    rf_final.fit(X, y, sample_weight=class_weights_final)
elif CAR_BALANCING:
    rf_final.fit(X, y, sample_weight=car_weights_final)
else:
    rf_final.fit(X, y)

importances_rf = pd.Series(rf_final.feature_importances_, index=feature_cols)
print("\nTop 20 features Random Forest:")
print(importances_rf.sort_values(ascending=False).head(20))

In [ ]:
xgb_final = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    n_jobs=-1,
    eval_metric="logloss",
    random_state=42,
)

if CLASS_BALANCING and CAR_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=class_weights_final)
elif CAR_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=car_weights_final)
else:
    xgb_final.fit(X, y_enc)

xgb_feature_scores = xgb_final.get_booster().get_score(importance_type="gain")

# Convert to pandas Series
importances_xgb = (
    pd.Series(xgb_feature_scores)
    .rename(index=lambda x: feature_cols[int(x[1:])])  # maps f0 → feature name
    .sort_values(ascending=False)
)

xgb_norm = importances_xgb / importances_xgb.sum()
print("\nTop 20 features XGBoost:")
print(xgb_norm.head(20))

In [ ]:
lgbm_final = LGBMClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

if CLASS_BALANCING and CAR_BALANCING:
    lgbm_final.fit(X, y_enc, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    lgbm_final.fit(X, y_enc, sample_weight=class_weights_final)
elif CAR_BALANCING:
    lgbm_final.fit(X, y_enc, sample_weight=car_weights_final)
else:
    lgbm_final.fit(X, y_enc)

importances_lgbm = pd.Series(lgbm_final.feature_importances_, index=feature_cols)
print("\nTop 20 features LightGBM:")
print(importances_lgbm.sort_values(ascending=False).head(20))